# Análisis de Riesgo de Mora en Clientes — XGBoost y SHAP



---

## SECCION 1: PREPROCESAMIENTO DE DATOS

### Descripción
Se importan las librerías necesarias para el análisis de datos y se carga la base de datos de financiación de clientes.

In [10]:
import pandas as pd
import numpy as np

In [11]:
df_financiacion = pd.read_excel('/Users/yedisoncuervo/Desktop/Proyecto-4_Analitica_III/Datos/BD taller clasificación (3).xlsx')
df_financiacion.head(5)

,Caso,Perfil,Estado,Edad,Genero,ScoreCrediticio,PorcentajeFinanciacion,Plazo,IngresoEstimado,Gastos,CapacidadDePago,ValorCuotaMensual,M3_30AC
0,1004991730,ASALARIADO,NUEVO,30,FEMENINO,748,0.6850,72,3289800.0,2430508.51,0.361093,2379693,0
1,1005097331,INDEPENDIENTE,NUEVO,46,MASCULINO,670,0.2783,60,2425440.0,1621788.08,0.948770,847046,0
2,1005120587,INDEPENDIENTE,USADO,39,MASCULINO,752,1.0000,60,30000000.0,3614018.63,12.009213,2197145,0
3,1005152562,ASALARIADO,USADO,38,FEMENINO,758,1.0000,84,1631331.0,1725244.99,-0.068706,1366896,0
4,1005153782,INDEPENDIENTE,NUEVO,60,FEMENINO,846,0.6596,72,20907400.0,3439341.88,13.004595,1343222,0


## PASO: 1 Información General de la Base de Datos

### Descripción
Se obtiene información general del dataset: estructura, tipos de datos, cantidad de registros y columnas.

In [12]:
df_financiacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21091 entries, 0 to 21090
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Caso                    21091 non-null  int64  
 1   Perfil                  21091 non-null  object 
 2   Estado                  21091 non-null  object 
 3   Edad                    21091 non-null  int64  
 4   Genero                  21091 non-null  object 
 5   ScoreCrediticio         21091 non-null  int64  
 6   PorcentajeFinanciacion  21091 non-null  float64
 7   Plazo                   21091 non-null  int64  
 8   IngresoEstimado         21063 non-null  float64
 9   Gastos                  21091 non-null  float64
 10  CapacidadDePago         21063 non-null  float64
 11  ValorCuotaMensual       21091 non-null  int64  
 12  M3_30AC                 21091 non-null  int64  
dtypes: float64(4), int64(6), object(3)
memory usage: 2.1+ MB


## PASO 2: Validación de Datos Faltantes, Nulos y Duplicados

### Descripción
Se realiza un análisis exhaustivo de la calidad de datos:
- **Datos nulos**: Se identifican y cuentan valores faltantes por variable
- **Datos duplicados**: Se detectan registros duplicados
- **Criterio de eliminación**: Si una columna tiene más del 40% de datos faltantes, será eliminada
- **Manejo de nulos**: Se eliminarán filas con valores faltantes si el porcentaje es bajo

In [13]:
# ──  Datos nulos ────────────────────────────────────────────────────────
print("DATOS NULOS")
nulos = df_financiacion.isnull().sum()
pct   = (nulos / len(df_financiacion) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct})
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]
if resumen_nulos.empty:
    print("No hay datos nulos.")
else:
    print(resumen_nulos)

DATOS NULOS
                 Nulos  Porcentaje (%)
IngresoEstimado     28            0.13
CapacidadDePago     28            0.13


### Tratamiento de Datos Nulos
Dado que el porcentaje de datos nulos en las variables (IngresosEstimado/CapacidadDePago) es de solo **0.13%**, se procede a **eliminar estas filas** de la base de datos, manteniendo la integridad de los datos.

In [14]:
# Eliminar filas con datos nulos
df_financiacion = df_financiacion.dropna()

In [15]:
# Verificar que quedaron cero nulos
print(f"Filas después de eliminar nulos: {len(df_financiacion)}")
print(f"Nulos restantes: {df_financiacion.isnull().sum().sum()}")

Filas después de eliminar nulos: 21063
Nulos restantes: 0


In [16]:
# ── 5. Datos duplicados ───────────────────────────────────────────────────
print("DATOS DUPLICADOS")
duplicados = df_financiacion.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")

DATOS DUPLICADOS
Filas duplicadas: 0


In [17]:
# ── 6. Estadísticas descriptivas ──────────────────────────────────────────
print("ESTADÍSTICAS DESCRIPTIVAS")
print(df_financiacion.describe())

ESTADÍSTICAS DESCRIPTIVAS
               Caso          Edad  ScoreCrediticio  PorcentajeFinanciacion  \
count  2.106300e+04  21063.000000     21063.000000            21063.000000   
mean   1.006178e+09     44.542563       782.361724                0.743488   
std    3.264530e+05     12.744980        85.314167                0.246626   
min    1.004992e+09     19.000000       343.000000                0.100000   
25%    1.005912e+09     34.000000       726.000000                0.552200   
50%    1.006159e+09     43.000000       783.000000                0.800000   
75%    1.006453e+09     54.000000       838.000000                1.000000   
max    1.006786e+09     75.000000       999.000000                1.067000   

              Plazo  IngresoEstimado        Gastos  CapacidadDePago  \
count  21063.000000     2.106300e+04  2.106300e+04     2.106300e+04   
mean      60.633101     5.018901e+06  1.142708e+08    -8.077237e+01   
std       12.497081     5.955286e+06  1.624658e+10     1.1

## Codificación de Variables Categóricas

### Descripción
Se convierten las variables categóricas binarias en formato numérico. Esta codificación se aplicará al dataset original antes de cualquier muestreo.

**Variables a codificar:**
- `Genero`: MASCULINO = 1, FEMENINO = 0
- `Perfil`: ASALARIADO = 1, INDEPENDIENTE = 0  
- `Estado`: NUEVO = 1, USADO = 0

In [18]:
# Verificar los valores únicos de variables categóricas antes de codificar
print("VALORES ÚNICOS EN VARIABLES CATEGÓRICAS")
print("=" * 50)
print(f"Genero: {df_financiacion['Genero'].unique()}")
print(f"Perfil: {df_financiacion['Perfil'].unique()}")
print(f"Estado: {df_financiacion['Estado'].unique()}")
print("=" * 50)

# ── Codificación de variables categóricas (binarias) ──────────────────────
df_financiacion['Genero']  = df_financiacion['Genero'].map({'MASCULINO': 1, 'FEMENINO': 0})
df_financiacion['Perfil']  = df_financiacion['Perfil'].map({'ASALARIADO': 1, 'INDEPENDIENTE': 0})
df_financiacion['Estado']  = df_financiacion['Estado'].map({'NUEVO': 1, 'USADO': 0})

print(" Codificación aplicada correctamente")
print(f"Verificación - Valores únicos después de codificar:")
print(f"Genero: {df_financiacion['Genero'].unique()}")
print(f"Perfil: {df_financiacion['Perfil'].unique()}")
print(f"Estado: {df_financiacion['Estado'].unique()}")

# Visualizar primeras filas del dataset codificado
print("Primeras filas del dataset después de codificar variables:")
df_financiacion.head()

# ── Eliminar variables que no aportan información ─────────────────────────
# Se elimina la columna 'Caso' que es solo un identificador de fila
df_financiacion = df_financiacion.drop(columns=['Caso'])

print("  Dataset preparado para muestreo y modelamiento")
print(f"\n Resumen final:")
print(f"   Total de registros: {len(df_financiacion)}")
print(f"   Total de variables: {df_financiacion.shape[1]}")
print(f"   Sin mora (Clase 0): {(df_financiacion['M3_30AC'] == 0).sum()}")
print(f"   Con mora (Clase 1): {(df_financiacion['M3_30AC'] == 1).sum()}")
print(f"   Proporción original: {(df_financiacion['M3_30AC'] == 0).sum() / (df_financiacion['M3_30AC'] == 1).sum():.1f}:1 (DESBALANCEADO)")
print(f"\n    El balanceo se aplicará a cada muestra en la Sección 2")

VALORES ÚNICOS EN VARIABLES CATEGÓRICAS
Genero: ['FEMENINO' 'MASCULINO']
Perfil: ['ASALARIADO' 'INDEPENDIENTE']
Estado: ['NUEVO' 'USADO']
 Codificación aplicada correctamente
Verificación - Valores únicos después de codificar:
Genero: [0 1]
Perfil: [1 0]
Estado: [1 0]
Primeras filas del dataset después de codificar variables:
  Dataset preparado para muestreo y modelamiento

 Resumen final:
   Total de registros: 21063
   Total de variables: 12
   Sin mora (Clase 0): 20228
   Con mora (Clase 1): 835
   Proporción original: 24.2:1 (DESBALANCEADO)

    El balanceo se aplicará a cada muestra en la Sección 2


In [19]:
# Guardar en Datos, la base de datos limpia y procesada
df_financiacion.to_csv('/Users/yedisoncuervo/Desktop/Proyecto-4_Analitica_III/Datos/BD_financiacion_limpia.csv', index=False)

---

## Resumen de la Preparación de Datos

### Transformaciones Realizadas:
- **Datos nulos**: Eliminadas filas con valores faltantes (0.13%)
- **Datos duplicados**: Verificación completada
- **Codificación**: Convertidas variables categóricas a numéricas
- **Limpieza**: Eliminada columna ID innecesaria
- **Generación**: Se genero un nuevo dataset con los datos procesados

### Dataset Preparado (SIN BALANCEAR AÚN):
- **Total de registros**: 21,330 observaciones
- **Total de variables**: 10 features + 1 target
- **Variable objetivo**: M3_30AC (Sin mora=0, Con mora=1)
- **Desbalance**: Proporción 24:1 (20,493 sin mora, 837 con mora)